# FCC-ee Impedance/Wake Model

In [ ]:

import sys
sys.path.append('/home/cantuono/IW2D/PYTHON_codes_and_scripts/General_Python_tools') #'path/to/your/IW2D'
sys.path.append('/home/cantuono/IW2D/PYTHON_codes_and_scripts/Impedance_lib_Python') ##'path/to/your/IW2D'

import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import matplotlib as mpl


In [ ]:
from fcc_ee_booster_pywit_model.fcc_ee_pywit_model import HEBModel
from lhc_pywit_model.lhc_model import LHCModel

## Beam Pipe retrival from PyWIT

- **Beam pipe:** Copper, circular cross-section, **30 mm radius**, coated with a **150 nm** non-evaporable getter (NEG) layer. Neg thickness to be confirmed

In [ ]:
repo_root = Path.cwd().parent
data_folder = repo_root / "fcc_ee_booster_pywit_model" /"data"

optics_settings_filename = (data_folder / "optics" / "FCCee_heb_modeZ.tfs")
elliptic_elements_settings_filename = (data_folder  / "elliptic_elements" /"FCC_ee_vz_elliptic_elements_settings_neg_150nm.json")

f_params_dict = {'start': 10,
                 'stop': 1e13,
                 'scan_type': 0,
                 'added': '1e-2 0.1 1 1e15',
                 'points_per_decade': 10,
                 'min_refine': 1e11,
                 'max_refine': 5e12,
                 'n_refine': 5000,
                 'sampling_exponent': 1e8,
                 }

additional_f_params = {'long_factor': 100,
                       'freq_lin_bisect': 1e11
                       }

z_params_dict = {'start': 0.01,
                 'stop': 1e13,
                 'scan_type': 2,
                 'added': ''
                 }

frequency_parameters_for_taper_rw = {
    'freq_start': 1e3,
    'freq_stop': 1e15,
    'num_points': 1000
}

### Creation model with PYWIT

In [ ]:
model_iw2d= HEBModel(
    energy=45.6e9, 
    optics_filename=optics_settings_filename, 
    elliptic_elements_settings_filename=elliptic_elements_settings_filename,
    f_params_dict=f_params_dict,
    z_params_dict=z_params_dict,
    additional_f_params=additional_f_params,
    f_cutoff_broadband=50e12,  
    compute_wake=False,
    beta_smooth_elements=['pipe_section_1'] 
)

In [ ]:
print('\nElements of the PyWIT model:\n')
print(model_iw2d.elements[0].name)

### Plots and export of impedances

In [ ]:
parent_dir = os.getcwd()
folder_path = os.path.join(parent_dir, 'Plots')
os.makedirs(folder_path, exist_ok=True)

In [ ]:
#%matplotlib ipympl

total_model = model_iw2d.elements[0]

disc_xdip = total_model.get_component('x1000').discretize(10**3, 1, 1e-1, 1e13, freq_precision_factor=0.1)[0]
disc_ydip = total_model.get_component('y0100').discretize(10**3, 1, 1e-1, 1e13, freq_precision_factor=0.1)[0]
disc_zlong = total_model.get_component('z0000').discretize(10**3, 1, 1e-1, 1e13, freq_precision_factor=0.1)[0]

plt.close('all')
parent_dir = os.getcwd()
folder_path = os.path.join(parent_dir, 'Plots')
os.makedirs(folder_path, exist_ok=True)

fntsz = 18
mpl.rcParams.update({'font.size': fntsz,
                     'xtick.labelsize': fntsz,
                     'ytick.labelsize': fntsz})

plt.figure(figsize=(7,5))
#plt.plot(disc_zlong[0], disc_zlong[1].real, '-', linewidth=2, label='Zz real')
plt.plot(disc_xdip[0], disc_xdip[1].real, '-', linewidth=2, label='Zx real')
plt.plot(disc_ydip[0], disc_ydip[1].real, '--', linewidth=2, label='Zy real')
plt.title(r'RW $Z_y^{dip}$ and $Z_x^{dip}$contibutions')
plt.xlabel('frequency [Hz]')
plt.ylabel(r'dipolar impedance [$\Omega/m$]')
plt.legend(loc='upper right')
plt.xlim(0,2e10)
#plt.ylim(-1e6,1e5)
plt.ylim(-1e6,5e6)
plt.tight_layout()
file_path = os.path.join(folder_path, 'RW_beam_pipe_V0_real.png')
plt.savefig(file_path)
plt.show()


plt.figure(figsize=(7,5))
#plt.plot(disc_zlong[0], disc_zlong[1].imag, '-', linewidth=2, label='Zz real')
plt.plot(disc_xdip[0], disc_xdip[1].imag, '-', linewidth=2, label='Zx imaginary')
plt.plot(disc_ydip[0], disc_ydip[1].imag, '--', linewidth=2, label='Zy imaginary')
plt.title(r'RW $Z_y^{dip}$ and $Z_x^{dip}$contibutions')
plt.xlabel('frequency [Hz]')
plt.ylabel(r'dipolar impedance [$\Omega/m$]')
plt.legend(loc='upper right')
plt.xlim(0,2e10)
plt.ylim(-1e6,0.1e8)
plt.tight_layout()
file_path = os.path.join(folder_path, 'RW_beam_pipe_V0_imag.png')
plt.savefig(file_path)
plt.show()

In [ ]:
parent_dir = os.getcwd()
txt_folder_path = os.path.join(parent_dir, 'Output_Impedance')
os.makedirs(txt_folder_path, exist_ok=True)

In [ ]:
comp = 'x1000'

freq_pywit_params = {  
    'freq_points': 10**4,
    'time_points': 1,
    'freq_start': 1e-1,
    'freq_stop': 2.4e10, 
    'freq_precision_factor': 0.1
}

# Get frequency range
tot_freq = model_iw2d.total.get_component(comp).discretize(
    10**5, 1, 1e-1, 2.4e10, 0.1
)[0][0]

# Export contributions for Component 0
component_1_real = model_iw2d.elements[0].get_component(comp).impedance(tot_freq).real
component_1_imag = model_iw2d.elements[0].get_component(comp).impedance(tot_freq).imag

# Save real part to file
component_1_real_file = os.path.join(txt_folder_path, 'Pipe_RW_V0_Zx_dip_Re.txt')
with open(component_1_real_file, 'w') as file:
    file.write(f"# Frequency (Hz)\tReal Part (Ohm/m)\n")
    for freq, real in zip(tot_freq, component_1_real):
        file.write(f"{freq}\t{real}\n")
print(f"Component 1 real part saved to {component_1_real_file}")

# Save imaginary part to file
component_1_imag_file = os.path.join(txt_folder_path, 'Pipe_RW_V0_Zx_dip_Im.txt')
with open(component_1_imag_file, 'w') as file:
    file.write(f"# Frequency (Hz)\tImaginary Part (Ohm/m)\n")
    for freq, imag in zip(tot_freq, component_1_imag):
        file.write(f"{freq}\t{imag}\n")
print(f"Component 1 imaginary part saved to {component_1_imag_file}")



### Tests for different NEG thicknesses

In [ ]:
import pandas as pd
import json

base_json_file = data_folder / "elliptic_elements" / "FCC_ee_vz_elliptic_elements_settings_neg_150nm.json"

neg_thicknesses = [1.5e-7, 3e-7, 5e-7]  # NEG thickness in meters
results = []

# Folder to save plots
folder_path = os.path.join(os.getcwd(), 'Plots')
os.makedirs(folder_path, exist_ok=True)

# -------------------------------
# Loop over NEG thicknesses
# -------------------------------
for t in neg_thicknesses:
    print(f"\n Running FCCEEModel with NEG thickness = {t*1e9:.1f} nm")

    # Load base JSON
    with open(base_json_file, "r") as f:
        elliptic_settings = json.load(f)

    # Modify NEG thickness
    elliptic_settings["pipe_section_1"]["layers"][0]["thickness"] = t

    # Save temporary JSON
    tmp_json_file = data_folder / f"elliptic_elements/tmp_neg_{int(t*1e9)}nm.json"
    with open(tmp_json_file, "w") as f:
        json.dump(elliptic_settings, f, indent=2)

    # -------------------------------
    # Call FCCEEModel
    # -------------------------------
    model = HEBModel(
        energy=45.6e9,
        optics_filename=optics_settings_filename,
        elliptic_elements_settings_filename=tmp_json_file,
        f_params_dict=f_params_dict,
        z_params_dict=z_params_dict,
        additional_f_params=additional_f_params,
        compute_wake=False,
        beta_smooth_elements=['pipe_section_1']
    )

    total_model = model.elements[0]  # beam pipe

    # discretize and cast points to int
    disc_xdip = total_model.get_component('x1000').discretize(int(1e3), 1, 1e-1, 1e13, freq_precision_factor=0.1)[0]
    disc_ydip = total_model.get_component('y0100').discretize(int(1e3), 1, 1e-1, 1e13, freq_precision_factor=0.1)[0]
    disc_zlong = total_model.get_component('z0000').discretize(int(1e3), 1, 1e-1, 1e13, freq_precision_factor=0.1)[0]

    # Store results for plotting
    results.append({
        "neg_thickness_m": t,
        "freq": disc_xdip[0],   # same for all
        "Zx": disc_xdip[1],
        "Zy": disc_ydip[1],
        "Zlong": disc_zlong[1]
    })



In [ ]:
# -------------------------------
# Plotting
# -------------------------------
plt.figure(figsize=(7,5))
for res in results:
    t_nm = res['neg_thickness_m']*1e9
    plt.loglog(res['freq'], np.real(res['Zx']), label=f'Zx, {t_nm:.0f} nm')
    plt.loglog(res['freq'], np.real(res['Zy']), '--', label=f'Zy, {t_nm:.0f} nm')

plt.xlabel('Frequency [Hz]')
plt.ylabel('Dipolar impedance [Ohm/m]')
plt.title('Impedance vs NEG thickness')
plt.legend()
plt.grid(True, which='both', ls='--')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
for res in results:
    t_nm = res['neg_thickness_m']*1e9
    plt.loglog(res['freq'], np.imag(res['Zx']), label=f'Zx, {t_nm:.0f} nm')
    plt.loglog(res['freq'], np.imag(res['Zy']), '--', label=f'Zy, {t_nm:.0f} nm')

plt.xlabel('Frequency [Hz]')
plt.ylabel('Dipolar impedance [Ohm/m]')
plt.title('Impedance vs NEG thickness')
plt.legend()
plt.grid(True, which='both', ls='--')
plt.tight_layout()
plt.show()